# 3D 网球轨迹分析

本 notebook 用于分析双摄像头网球追踪系统的 3D 轨迹质量，测试轨迹拟合与噪声过滤方法。

## 流程
1. **数据准备** — 加载 cam66/cam68 的检测数据（含GT和模型输出），加载单应性矩阵
2. **2D → 世界坐标** — 通过单应性矩阵将像素坐标投影到地面平面
3. **三角测量 → 3D** — 双摄像头射线求交，得到 3D 坐标 (x, y, z)
4. **轨迹可视化** — 原始 3D 轨迹、2D 投影、时间序列
5. **轨迹分段** — 按击球/弹地事件切分成独立飞行段
6. **段内滤波** — 测试中值滤波、Savitzky-Golay、抛物线拟合等方法
7. **对比评估** — 滤波前后 vs GT 的误差对比

---
## 1. 环境与配置

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import medfilt, savgol_filter
from scipy.interpolate import CubicSpline

# 项目根目录（从 notebooks/ 上一级）
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# 可复用项目模块
from app.pipeline.homography import HomographyTransformer
from app.triangulation import triangulate

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

In [ ]:
# ===== 路径配置 =====
# GT + 混合数据目录（手动标注覆盖了部分模型输出）
CAM66_DIR = PROJECT_ROOT / 'uploads' / 'cam66_20260307_173403_2min'
CAM68_DIR = PROJECT_ROOT / 'uploads' / 'cam68_20260307_173403_2min'

# 纯模型输出目录（export_labelme 生成，不含 GT）
CAM66_MODEL_DIR = PROJECT_ROOT / 'exports' / 'cam66_model'
CAM68_MODEL_DIR = PROJECT_ROOT / 'exports' / 'cam68_model'

HOMOGRAPHY_PATH = str(PROJECT_ROOT / 'src' / 'homography_matrices.json')

# 帧范围：GT 校验范围
FRAME_START = 0
FRAME_END = 341

# 相机 3D 位置 (来自 config.yaml)
CAM66_POS = [3.7657, -15.7273, 7.0]
CAM68_POS = [5.8058, 45.0351, 7.0]

# 球场尺寸
COURT_WIDTH = 8.23   # m
COURT_LENGTH = 23.77 # m
NET_Y = 11.885       # m

print(f'数据范围: 帧 {FRAME_START}-{FRAME_END}')
print(f'cam66 GT目录: {CAM66_DIR}')
print(f'cam68 GT目录: {CAM68_DIR}')
print(f'cam66 模型目录: {CAM66_MODEL_DIR}')
print(f'cam68 模型目录: {CAM68_MODEL_DIR}')

---
## 2. 数据加载

In [ ]:
def load_labelme_detections(data_dir: Path, frame_start: int, frame_end: int) -> pd.DataFrame:
    """加载 LabelMe JSON 标注，返回 DataFrame。
    
    区分 GT（score=None）和模型输出（score=float）。
    每帧取第一个 shape（top-1 blob 或 GT 标注）。
    """
    rows = []
    for i in range(frame_start, frame_end + 1):
        jf = data_dir / f'{i:05d}.json'
        if not jf.exists():
            continue
        with open(jf, encoding='utf-8') as f:
            data = json.load(f)
        
        shapes = data.get('shapes', [])
        if not shapes:
            rows.append({'frame': i, 'pixel_x': np.nan, 'pixel_y': np.nan,
                        'is_gt': False, 'score': np.nan, 'has_detection': False})
            continue
        
        s = shapes[0]
        px, py = s['points'][0]
        score = s.get('score')
        is_gt = score is None
        
        rows.append({
            'frame': i,
            'pixel_x': px,
            'pixel_y': py,
            'is_gt': is_gt,
            'score': score if score is not None else np.nan,
            'has_detection': True,
        })
    
    return pd.DataFrame(rows).set_index('frame')


# 加载两个摄像头的数据
df_cam66 = load_labelme_detections(CAM66_DIR, FRAME_START, FRAME_END)
df_cam68 = load_labelme_detections(CAM68_DIR, FRAME_START, FRAME_END)

print(f'cam66: {len(df_cam66)} 帧, {df_cam66["has_detection"].sum()} 有检测, {df_cam66["is_gt"].sum()} 是GT')
print(f'cam68: {len(df_cam68)} 帧, {df_cam68["has_detection"].sum()} 有检测, {df_cam68["is_gt"].sum()} 是GT')

In [ ]:
# 快速预览数据
print('=== cam66 前10帧 ===')
display(df_cam66.head(10))
print('\n=== cam68 前10帧 ===')
display(df_cam68.head(10))

---
## 3. 单应性投影：像素 → 世界坐标

In [ ]:
# 加载单应性矩阵
homo_cam66 = HomographyTransformer(HOMOGRAPHY_PATH, 'cam66')
homo_cam68 = HomographyTransformer(HOMOGRAPHY_PATH, 'cam68')


def add_world_coords(df: pd.DataFrame, homo: HomographyTransformer) -> pd.DataFrame:
    """将像素坐标通过单应性投影到世界坐标。"""
    world_x, world_y = [], []
    for _, row in df.iterrows():
        if row['has_detection'] and not np.isnan(row['pixel_x']):
            wx, wy = homo.pixel_to_world(row['pixel_x'], row['pixel_y'])
            world_x.append(wx)
            world_y.append(wy)
        else:
            world_x.append(np.nan)
            world_y.append(np.nan)
    df['world_x'] = world_x
    df['world_y'] = world_y
    return df


df_cam66 = add_world_coords(df_cam66, homo_cam66)
df_cam68 = add_world_coords(df_cam68, homo_cam68)

print('cam66 世界坐标范围:')
print(f'  world_x: [{df_cam66["world_x"].min():.1f}, {df_cam66["world_x"].max():.1f}] m')
print(f'  world_y: [{df_cam66["world_y"].min():.1f}, {df_cam66["world_y"].max():.1f}] m')
print('cam68 世界坐标范围:')
print(f'  world_x: [{df_cam68["world_x"].min():.1f}, {df_cam68["world_x"].max():.1f}] m')
print(f'  world_y: [{df_cam68["world_y"].min():.1f}, {df_cam68["world_y"].max():.1f}] m')

---
## 4. 三角测量：双摄像头 → 3D 坐标

**原理**：每个摄像头的检测通过单应性投影到地面平面 (z=0)，然后从摄像头 3D 位置向地面投影点作射线。
两条射线在空间中的最近点中点即为球的 3D 位置。

```
cam66 (3.77, -15.73, 7.0)          cam68 (5.81, 45.04, 7.0)
         \                              /
          \  ray1              ray2    /
           \                        /
            \    ● 球 (x,y,z)     /
             \  /            \  /
              \/              \/
    ─────── ground1 ───── ground2 ──────
              z = 0 地面平面
```

In [ ]:
def compute_3d_trajectory(df66: pd.DataFrame, df68: pd.DataFrame) -> pd.DataFrame:
    """对两个摄像头都有检测的帧，执行三角测量得到 3D 坐标。"""
    rows = []
    common_frames = df66.index.intersection(df68.index)
    
    for fi in common_frames:
        r66 = df66.loc[fi]
        r68 = df68.loc[fi]
        
        # 两个摄像头都必须有检测
        if not (r66['has_detection'] and r68['has_detection']):
            continue
        if np.isnan(r66['world_x']) or np.isnan(r68['world_x']):
            continue
        
        # 三角测量
        x3d, y3d, z3d = triangulate(
            (r66['world_x'], r66['world_y']),
            (r68['world_x'], r68['world_y']),
            CAM66_POS,
            CAM68_POS,
        )
        
        # 数据来源标记
        source = 'both_gt' if (r66['is_gt'] and r68['is_gt']) else \
                 'one_gt' if (r66['is_gt'] or r68['is_gt']) else 'both_model'
        
        rows.append({
            'frame': fi,
            'x': x3d, 'y': y3d, 'z': z3d,
            'cam66_wx': r66['world_x'], 'cam66_wy': r66['world_y'],
            'cam68_wx': r68['world_x'], 'cam68_wy': r68['world_y'],
            'cam66_px': r66['pixel_x'], 'cam66_py': r66['pixel_y'],
            'cam68_px': r68['pixel_x'], 'cam68_py': r68['pixel_y'],
            'source': source,
        })
    
    return pd.DataFrame(rows).set_index('frame')


df_3d = compute_3d_trajectory(df_cam66, df_cam68)
print(f'3D 轨迹: {len(df_3d)} 帧')
print(f'  数据来源: {df_3d["source"].value_counts().to_dict()}')
print(f'  x: [{df_3d["x"].min():.2f}, {df_3d["x"].max():.2f}] m')
print(f'  y: [{df_3d["y"].min():.2f}, {df_3d["y"].max():.2f}] m')
print(f'  z: [{df_3d["z"].min():.2f}, {df_3d["z"].max():.2f}] m')

---
## 5. 原始轨迹可视化

In [ ]:
def draw_court(ax, top_down=True):
    """在 ax 上绘制网球场线。"""
    W, L = COURT_WIDTH, COURT_LENGTH
    # 边线
    if top_down:
        ax.plot([0, W, W, 0, 0], [0, 0, L, L, 0], 'w-', linewidth=1.5)
        ax.plot([0, W], [NET_Y, NET_Y], 'w--', linewidth=1, alpha=0.7)  # 球网
        ax.plot([0, W], [5.485, 5.485], 'w-', linewidth=0.8, alpha=0.5)  # 近发球线
        ax.plot([0, W], [18.285, 18.285], 'w-', linewidth=0.8, alpha=0.5)  # 远发球线
        ax.plot([W/2, W/2], [5.485, 18.285], 'w-', linewidth=0.8, alpha=0.5)  # 中线
        ax.set_facecolor('#2d5c2d')
    else:
        ax.plot([0, W, W, 0, 0], [0, 0, L, L, 0], 'g-', linewidth=1)
        ax.axhline(NET_Y, color='gray', linestyle='--', linewidth=0.8)


# --- 图1: 俯视图 (x, y) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 俯视图
ax = axes[0]
draw_court(ax)
sc = ax.scatter(df_3d['x'], df_3d['y'], c=df_3d.index, cmap='coolwarm', s=8, zorder=5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('俯视图 (X-Y)')
ax.set_aspect('equal')
plt.colorbar(sc, ax=ax, label='帧号')

# Y-Z 侧视图（高度）
ax = axes[1]
ax.scatter(df_3d['y'], df_3d['z'], c=df_3d.index, cmap='coolwarm', s=8)
ax.axhline(0, color='brown', linewidth=1, label='地面')
ax.axhline(0.914, color='gray', linestyle='--', linewidth=0.8, label='球网高度')
ax.axvline(NET_Y, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Y (m)')
ax.set_ylabel('Z (m) 高度')
ax.set_title('侧视图 (Y-Z)')
ax.legend(fontsize=8)

# X-Z 正视图
ax = axes[2]
ax.scatter(df_3d['x'], df_3d['z'], c=df_3d.index, cmap='coolwarm', s=8)
ax.axhline(0, color='brown', linewidth=1)
ax.axhline(0.914, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('X (m)')
ax.set_ylabel('Z (m) 高度')
ax.set_title('正视图 (X-Z)')

plt.tight_layout()
plt.show()

In [ ]:
# --- 图2: 3D 交互式轨迹 (plotly) ---
import plotly.graph_objects as go

fig3d = go.Figure()

# 球轨迹点（按帧号着色）
fig3d.add_trace(go.Scatter3d(
    x=df_3d['x'], y=df_3d['y'], z=df_3d['z'],
    mode='markers',
    marker=dict(size=3, color=df_3d.index, colorscale='RdBu', colorbar=dict(title='帧号'), opacity=0.8),
    text=[f'帧{f}<br>x={r["x"]:.2f} y={r["y"]:.2f} z={r["z"]:.2f}' for f, r in df_3d.iterrows()],
    hoverinfo='text',
    name='球轨迹',
))

# 球场地面
court_x = [0, COURT_WIDTH, COURT_WIDTH, 0, 0]
court_y_line = [0, 0, COURT_LENGTH, COURT_LENGTH, 0]
fig3d.add_trace(go.Scatter3d(
    x=court_x, y=court_y_line, z=[0]*5,
    mode='lines', line=dict(color='green', width=3),
    name='球场边线', showlegend=True,
))

# 球网
fig3d.add_trace(go.Scatter3d(
    x=[0, COURT_WIDTH, COURT_WIDTH, 0, 0],
    y=[NET_Y]*5,
    z=[0, 0, 0.914, 0.914, 0],
    mode='lines', line=dict(color='gray', width=2),
    name='球网',
))

# 相机位置
fig3d.add_trace(go.Scatter3d(
    x=[CAM66_POS[0]], y=[CAM66_POS[1]], z=[CAM66_POS[2]],
    mode='markers+text', marker=dict(size=8, color='blue', symbol='diamond'),
    text=['cam66'], textposition='top center', name='cam66',
))
fig3d.add_trace(go.Scatter3d(
    x=[CAM68_POS[0]], y=[CAM68_POS[1]], z=[CAM68_POS[2]],
    mode='markers+text', marker=dict(size=8, color='red', symbol='diamond'),
    text=['cam68'], textposition='top center', name='cam68',
))

fig3d.update_layout(
    title='3D 球轨迹（可旋转/缩放）',
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        aspectmode='data',
    ),
    width=900, height=700,
)
fig3d.show()

In [ ]:
# --- 图3: 时间序列 x, y, z ---
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

for ax, col, label, color in zip(axes, ['x', 'y', 'z'],
                                  ['X (球场宽度方向)', 'Y (球场长度方向)', 'Z (高度)'],
                                  ['tab:blue', 'tab:orange', 'tab:green']):
    ax.plot(df_3d.index, df_3d[col], '.', markersize=3, color=color, alpha=0.6, label='原始')
    ax.set_ylabel(f'{label} (m)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)

axes[0].set_title('3D 坐标时间序列 (原始)')
axes[-1].set_xlabel('帧号')
plt.tight_layout()
plt.show()

---
## 6. 噪声分析

计算帧间速度和加速度，识别异常跳点和轨迹分段点。

In [ ]:
# 计算帧间 3D 速度（m/帧 → m/s at 25fps）
FPS = 25.0
dt = 1.0 / FPS

df_3d['dx'] = df_3d['x'].diff()
df_3d['dy'] = df_3d['y'].diff()
df_3d['dz'] = df_3d['z'].diff()
df_3d['d_frame'] = df_3d.index.to_series().diff()  # 帧间隔（可能有跳帧）
df_3d['dist_3d'] = np.sqrt(df_3d['dx']**2 + df_3d['dy']**2 + df_3d['dz']**2)
df_3d['speed_ms'] = df_3d['dist_3d'] / (df_3d['d_frame'] * dt)  # m/s

print('帧间速度统计 (m/s):')
print(df_3d['speed_ms'].describe())
print(f'\n网球最高速度参考: ~70 m/s (发球)')
print(f'超过 70 m/s 的帧数: {(df_3d["speed_ms"] > 70).sum()}')
print(f'超过 100 m/s 的帧数: {(df_3d["speed_ms"] > 100).sum()} (明显异常)')

In [ ]:
# --- 图4: 速度时间序列 ---
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df_3d.index, df_3d['speed_ms'], '.-', markersize=3, linewidth=0.5)
ax.axhline(70, color='red', linestyle='--', linewidth=0.8, label='网球最高速度 ~70m/s')
ax.set_xlabel('帧号')
ax.set_ylabel('速度 (m/s)')
ax.set_title('帧间 3D 速度')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. 轨迹分段

按照速度突变（击球/弹地）将轨迹切分为独立飞行段。
每段内部近似恒定加速度（重力），可以独立进行滤波和拟合。

In [ ]:
def segment_trajectory(df: pd.DataFrame, 
                       speed_threshold: float = 50.0,
                       gap_threshold: int = 5,
                       min_segment_len: int = 4) -> list[pd.DataFrame]:
    """将 3D 轨迹切分为连续飞行段。
    
    分段条件:
    1. 帧间速度超过 speed_threshold（可能是误检或击球）
    2. 帧间隔超过 gap_threshold（检测中断）
    
    返回: list of DataFrame，每段一个。
    """
    # 找分段点
    cut_points = []
    for i in range(1, len(df)):
        speed = df['speed_ms'].iloc[i]
        d_frame = df['d_frame'].iloc[i]
        if (not np.isnan(speed) and speed > speed_threshold) or \
           (not np.isnan(d_frame) and d_frame > gap_threshold):
            cut_points.append(i)
    
    # 按分段点切分
    segments = []
    prev = 0
    for cp in cut_points:
        seg = df.iloc[prev:cp]
        if len(seg) >= min_segment_len:
            segments.append(seg.copy())
        prev = cp
    # 最后一段
    seg = df.iloc[prev:]
    if len(seg) >= min_segment_len:
        segments.append(seg.copy())
    
    return segments


segments = segment_trajectory(df_3d, speed_threshold=50.0, gap_threshold=5)
print(f'共切分为 {len(segments)} 段')
for i, seg in enumerate(segments):
    f0, f1 = seg.index[0], seg.index[-1]
    print(f'  段{i}: 帧 {f0}-{f1} ({len(seg)}帧), '
          f'z=[{seg["z"].min():.2f}, {seg["z"].max():.2f}]m')

In [ ]:
# --- 图5: 分段可视化 ---
fig, ax = plt.subplots(figsize=(16, 5))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(segments), 10)))

for i, seg in enumerate(segments):
    ax.plot(seg.index, seg['z'], '.-', color=colors[i % 10], markersize=4,
            linewidth=1, label=f'段{i} ({seg.index[0]}-{seg.index[-1]})')

ax.axhline(0, color='brown', linewidth=1, label='地面')
ax.axhline(0.914, color='gray', linestyle='--', linewidth=0.8, label='球网高度')
ax.set_xlabel('帧号')
ax.set_ylabel('Z (m) 高度')
ax.set_title('轨迹分段 — Z 高度')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. 段内滤波方法对比

在每个飞行段内独立应用不同的滤波方法，对比效果。

**核心原则：滤波不反馈回检测**（详见 docs/trajectory_filtering.md）

In [ ]:
def apply_filters(seg: pd.DataFrame) -> pd.DataFrame:
    """对一个轨迹段应用多种滤波方法，返回包含所有结果的 DataFrame。"""
    result = seg[['x', 'y', 'z']].copy()
    n = len(result)
    
    for col in ['x', 'y', 'z']:
        raw = result[col].values
        
        # 1. 中值滤波 (kernel=3)
        if n >= 3:
            result[f'{col}_median'] = medfilt(raw, kernel_size=min(5, n if n % 2 == 1 else n - 1))
        else:
            result[f'{col}_median'] = raw
        
        # 2. Savitzky-Golay (window=7, poly=2)
        win = min(7, n if n % 2 == 1 else n - 1)
        if win >= 3:
            result[f'{col}_sg'] = savgol_filter(raw, window_length=win, polyorder=min(2, win - 1))
        else:
            result[f'{col}_sg'] = raw
        
        # 3. 抛物线拟合 (z轴用2次，x/y用1次)
        frames = np.arange(n)
        poly_order = 2 if col == 'z' else 1
        if n > poly_order + 1:
            coeffs = np.polyfit(frames, raw, poly_order)
            result[f'{col}_poly'] = np.polyval(coeffs, frames)
        else:
            result[f'{col}_poly'] = raw
    
    return result


# 对每段应用滤波
filtered_segments = [apply_filters(seg) for seg in segments]
print(f'已对 {len(filtered_segments)} 段应用滤波')

In [ ]:
# --- 图6: 选择一个典型段展示滤波效果 ---
# 找到一个帧数适中的段
seg_idx = max(range(len(segments)), key=lambda i: len(segments[i]))
seg_raw = segments[seg_idx]
seg_filt = filtered_segments[seg_idx]

print(f'展示段 {seg_idx}: 帧 {seg_raw.index[0]}-{seg_raw.index[-1]} ({len(seg_raw)}帧)')

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

for ax, col, label in zip(axes, ['x', 'y', 'z'], ['X', 'Y', 'Z']):
    frames = seg_filt.index
    ax.plot(frames, seg_filt[col], 'k.', markersize=5, label='原始', alpha=0.5)
    ax.plot(frames, seg_filt[f'{col}_median'], '-', linewidth=1.5, label='中值滤波', alpha=0.8)
    ax.plot(frames, seg_filt[f'{col}_sg'], '-', linewidth=1.5, label='Savitzky-Golay', alpha=0.8)
    ax.plot(frames, seg_filt[f'{col}_poly'], '--', linewidth=1.5, label='多项式拟合', alpha=0.8)
    ax.set_ylabel(f'{label} (m)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[0].set_title(f'段 {seg_idx} 滤波方法对比 (帧 {seg_raw.index[0]}-{seg_raw.index[-1]})')
axes[-1].set_xlabel('帧号')
plt.tight_layout()
plt.show()

In [ ]:
# --- 图7: 滤波后的 3D 轨迹（选一个段，plotly 交互式）---
import plotly.graph_objects as go

fig7 = go.Figure()

# 原始点
fig7.add_trace(go.Scatter3d(
    x=seg_filt['x'], y=seg_filt['y'], z=seg_filt['z'],
    mode='markers+lines',
    marker=dict(size=3, color='black', opacity=0.4),
    line=dict(color='black', width=1, dash='dash'),
    text=[f'帧{f}' for f in seg_filt.index],
    hoverinfo='text',
    name='原始',
))

# SG 滤波
fig7.add_trace(go.Scatter3d(
    x=seg_filt['x_sg'], y=seg_filt['y_sg'], z=seg_filt['z_sg'],
    mode='lines',
    line=dict(color='red', width=4),
    name='Savitzky-Golay',
))

# 多项式拟合
fig7.add_trace(go.Scatter3d(
    x=seg_filt['x_poly'], y=seg_filt['y_poly'], z=seg_filt['z_poly'],
    mode='lines',
    line=dict(color='blue', width=4, dash='dash'),
    name='多项式拟合',
))

# 中值滤波
fig7.add_trace(go.Scatter3d(
    x=seg_filt['x_median'], y=seg_filt['y_median'], z=seg_filt['z_median'],
    mode='lines',
    line=dict(color='orange', width=3),
    name='中值滤波',
))

fig7.update_layout(
    title=f'段 {seg_idx} 3D 滤波对比（帧 {seg_raw.index[0]}-{seg_raw.index[-1]}）',
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        aspectmode='data',
    ),
    width=900, height=600,
)
fig7.show()

---
## 9. 噪声异常点识别

通过速度阈值和残差分析，识别哪些 3D 点是噪声/误检。

In [ ]:
def identify_outliers(seg_raw: pd.DataFrame, seg_filt: pd.DataFrame,
                      residual_threshold: float = 0.5) -> pd.Series:
    """通过滤波残差识别异常点。
    
    如果某帧的原始值与 SG 滤波值偏差超过阈值，标记为异常。
    """
    residual = np.sqrt(
        (seg_filt['x'] - seg_filt['x_sg'])**2 +
        (seg_filt['y'] - seg_filt['y_sg'])**2 +
        (seg_filt['z'] - seg_filt['z_sg'])**2
    )
    return residual > residual_threshold


# 对最长段做异常点检测
outliers = identify_outliers(seg_raw, seg_filt, residual_threshold=0.5)
n_outliers = outliers.sum()
print(f'段 {seg_idx}: {n_outliers}/{len(outliers)} 个异常点 ({n_outliers/len(outliers)*100:.1f}%)')

if n_outliers > 0:
    print('异常帧:', seg_filt.index[outliers].tolist())

In [ ]:
# --- 图8: 异常点标注 ---
fig, ax = plt.subplots(figsize=(16, 5))

# 正常点
normal = ~outliers
ax.plot(seg_filt.index[normal], seg_filt['z'][normal], 'b.', markersize=6, label='正常')
# 异常点
if n_outliers > 0:
    ax.plot(seg_filt.index[outliers], seg_filt['z'][outliers], 'rx', markersize=10, 
            markeredgewidth=2, label='异常点')
# 滤波曲线
ax.plot(seg_filt.index, seg_filt['z_sg'], 'g-', linewidth=2, label='SG 滤波', alpha=0.7)

ax.axhline(0, color='brown', linewidth=1)
ax.set_xlabel('帧号')
ax.set_ylabel('Z (m)')
ax.set_title(f'段 {seg_idx} 异常点检测')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. 统计摘要

In [ ]:
# 全局统计
print('=' * 60)
print('3D 轨迹分析摘要')
print('=' * 60)
print(f'分析帧范围: {FRAME_START}-{FRAME_END}')
print(f'双摄像头配对帧数: {len(df_3d)}')
print(f'轨迹分段数: {len(segments)}')
print(f'\n3D 坐标范围:')
print(f'  X: [{df_3d["x"].min():.2f}, {df_3d["x"].max():.2f}] m  (球场宽 {COURT_WIDTH}m)')
print(f'  Y: [{df_3d["y"].min():.2f}, {df_3d["y"].max():.2f}] m  (球场长 {COURT_LENGTH}m)')
print(f'  Z: [{df_3d["z"].min():.2f}, {df_3d["z"].max():.2f}] m')
print(f'\n速度统计 (m/s):')
print(f'  中位数: {df_3d["speed_ms"].median():.1f}')
print(f'  平均值: {df_3d["speed_ms"].mean():.1f}')
print(f'  最大值: {df_3d["speed_ms"].max():.1f}')
print(f'  超过70m/s (疑似异常): {(df_3d["speed_ms"] > 70).sum()} 帧')

# 各段统计
print(f'\n各段详情:')
print(f'{"段":>3} {"帧范围":>12} {"帧数":>5} {"z_min":>7} {"z_max":>7} {"avg_speed":>10}')
print('-' * 55)
for i, seg in enumerate(segments):
    print(f'{i:3d} {seg.index[0]:5d}-{seg.index[-1]:<5d} {len(seg):5d} '
          f'{seg["z"].min():7.2f} {seg["z"].max():7.2f} '
          f'{seg["speed_ms"].mean():8.1f} m/s')

---
## 11. GT vs 模型输出对比

加载纯模型输出（`exports/cam66_model/`, `exports/cam68_model/`），与 GT 标注做逐帧像素误差对比。

**数据说明**：
- GT 数据在 `uploads/` 目录中，`score=None` 的帧是手动标注
- 模型输出在 `exports/` 目录中，由 `export_labelme` 独立跑出（threshold=0.3/0.35）
- 对比只在 GT 帧上进行，计算模型预测 vs GT 的像素距离

In [ ]:
# 加载纯模型输出
df_model_cam66 = load_labelme_detections(CAM66_MODEL_DIR, FRAME_START, FRAME_END)
df_model_cam68 = load_labelme_detections(CAM68_MODEL_DIR, FRAME_START, FRAME_END)

print(f'模型输出 cam66: {len(df_model_cam66)} 帧, {df_model_cam66["has_detection"].sum()} 有检测')
print(f'模型输出 cam68: {len(df_model_cam68)} 帧, {df_model_cam68["has_detection"].sum()} 有检测')

# 提取 GT 帧（从 uploads 目录）
gt_cam66 = df_cam66[df_cam66['is_gt'] & df_cam66['has_detection']].copy()
gt_cam68 = df_cam68[df_cam68['is_gt'] & df_cam68['has_detection']].copy()
print(f'\nGT 帧数: cam66={len(gt_cam66)}, cam68={len(gt_cam68)}')

In [ ]:
def compare_gt_vs_model(gt_df: pd.DataFrame, model_df: pd.DataFrame, cam_name: str) -> pd.DataFrame:
    """逐帧对比 GT 与模型输出的像素误差。
    
    Returns:
        DataFrame with columns: gt_px, gt_py, model_px, model_py, pixel_error, status
    """
    rows = []
    for fi in gt_df.index:
        gt_px = gt_df.loc[fi, 'pixel_x']
        gt_py = gt_df.loc[fi, 'pixel_y']
        
        if fi not in model_df.index:
            rows.append({'frame': fi, 'gt_px': gt_px, 'gt_py': gt_py,
                        'model_px': np.nan, 'model_py': np.nan,
                        'pixel_error': np.nan, 'status': 'model_no_frame'})
            continue
        
        model_row = model_df.loc[fi]
        if not model_row['has_detection']:
            rows.append({'frame': fi, 'gt_px': gt_px, 'gt_py': gt_py,
                        'model_px': np.nan, 'model_py': np.nan,
                        'pixel_error': np.nan, 'status': 'model_miss'})
            continue
        
        model_px = model_row['pixel_x']
        model_py = model_row['pixel_y']
        err = np.sqrt((gt_px - model_px)**2 + (gt_py - model_py)**2)
        
        # 误差判定: <10px 正确, 10-50px 偏差, >50px 错误
        if err < 10:
            status = 'correct'
        elif err < 50:
            status = 'offset'
        else:
            status = 'wrong'
        
        rows.append({
            'frame': fi, 'gt_px': gt_px, 'gt_py': gt_py,
            'model_px': model_px, 'model_py': model_py,
            'pixel_error': err, 'status': status,
        })
    
    return pd.DataFrame(rows).set_index('frame')


# 对比两个摄像头
cmp_cam66 = compare_gt_vs_model(gt_cam66, df_model_cam66, 'cam66')
cmp_cam68 = compare_gt_vs_model(gt_cam68, df_model_cam68, 'cam68')

for name, cmp in [('cam66', cmp_cam66), ('cam68', cmp_cam68)]:
    print(f'\n=== {name} GT vs 模型 ===')
    print(f'  GT 帧数: {len(cmp)}')
    print(f'  状态分布: {cmp["status"].value_counts().to_dict()}')
    detected = cmp[cmp['status'].isin(['correct', 'offset', 'wrong'])]
    if len(detected) > 0:
        print(f'  像素误差 (有检测帧):')
        print(f'    平均: {detected["pixel_error"].mean():.1f} px')
        print(f'    中位: {detected["pixel_error"].median():.1f} px')
        print(f'    最大: {detected["pixel_error"].max():.1f} px')
        print(f'    <10px: {(detected["pixel_error"] < 10).sum()}/{len(detected)}')

In [ ]:
# --- 图9: GT vs 模型 像素误差分布 ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, name, cmp in zip(axes, ['cam66', 'cam68'], [cmp_cam66, cmp_cam68]):
    detected = cmp[cmp['pixel_error'].notna()]
    if len(detected) == 0:
        ax.text(0.5, 0.5, '无数据', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{name}: 无对比数据')
        continue
    
    ax.bar(detected.index, detected['pixel_error'], 
           color=['green' if s == 'correct' else 'orange' if s == 'offset' else 'red' 
                  for s in detected['status']],
           alpha=0.7)
    ax.axhline(10, color='green', linestyle='--', linewidth=0.8, label='<10px (正确)')
    ax.axhline(50, color='red', linestyle='--', linewidth=0.8, label='<50px (偏差)')
    
    # 标注漏检帧
    missed = cmp[cmp['status'].isin(['model_miss', 'model_no_frame'])]
    for fi in missed.index:
        ax.axvline(fi, color='gray', linewidth=0.5, alpha=0.3)
    
    ax.set_xlabel('帧号')
    ax.set_ylabel('像素误差 (px)')
    ax.set_title(f'{name}: GT vs 模型 像素误差 ({len(detected)}帧有检测, {len(missed)}帧漏检)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- 图10: GT vs 模型 3D 轨迹对比 ---
# 用模型输出重新计算 3D 轨迹
df_model_cam66_w = add_world_coords(df_model_cam66.copy(), homo_cam66)
df_model_cam68_w = add_world_coords(df_model_cam68.copy(), homo_cam68)
df_3d_model = compute_3d_trajectory(df_model_cam66_w, df_model_cam68_w)

# 用 GT 计算 3D 轨迹（只取 GT 帧）
gt66_only = df_cam66[df_cam66['is_gt']].copy()
gt68_only = df_cam68[df_cam68['is_gt']].copy()
df_3d_gt = compute_3d_trajectory(gt66_only, gt68_only)

print(f'模型 3D 轨迹: {len(df_3d_model)} 帧')
print(f'GT 3D 轨迹: {len(df_3d_gt)} 帧')

# 在共同帧上计算 3D 误差
common = df_3d_gt.index.intersection(df_3d_model.index)
if len(common) > 0:
    err_3d = np.sqrt(
        (df_3d_gt.loc[common, 'x'] - df_3d_model.loc[common, 'x'])**2 +
        (df_3d_gt.loc[common, 'y'] - df_3d_model.loc[common, 'y'])**2 +
        (df_3d_gt.loc[common, 'z'] - df_3d_model.loc[common, 'z'])**2
    )
    print(f'\n3D 误差 (GT vs 模型, {len(common)} 共同帧):')
    print(f'  平均: {err_3d.mean():.3f} m')
    print(f'  中位: {err_3d.median():.3f} m')
    print(f'  最大: {err_3d.max():.3f} m')
    print(f'  <0.5m: {(err_3d < 0.5).sum()}/{len(common)}')
else:
    print('无共同 GT 帧（两个摄像头都有 GT 标注的帧）')

In [ ]:
# --- 图11: 3D 交互式 GT vs 模型轨迹对比 ---
import plotly.graph_objects as go

fig11 = go.Figure()

# 模型 3D 轨迹
fig11.add_trace(go.Scatter3d(
    x=df_3d_model['x'], y=df_3d_model['y'], z=df_3d_model['z'],
    mode='markers',
    marker=dict(size=2, color='blue', opacity=0.5),
    text=[f'帧{f} (模型)<br>x={r["x"]:.2f} y={r["y"]:.2f} z={r["z"]:.2f}'
          for f, r in df_3d_model.iterrows()],
    hoverinfo='text',
    name='模型输出',
))

# GT 3D 轨迹
if len(df_3d_gt) > 0:
    fig11.add_trace(go.Scatter3d(
        x=df_3d_gt['x'], y=df_3d_gt['y'], z=df_3d_gt['z'],
        mode='markers',
        marker=dict(size=5, color='red', symbol='x', opacity=0.9),
        text=[f'帧{f} (GT)<br>x={r["x"]:.2f} y={r["y"]:.2f} z={r["z"]:.2f}'
              for f, r in df_3d_gt.iterrows()],
        hoverinfo='text',
        name='GT 真值',
    ))

# 误差连线（GT 到模型）
if len(common) > 0:
    for fi in common:
        fig11.add_trace(go.Scatter3d(
            x=[df_3d_gt.loc[fi, 'x'], df_3d_model.loc[fi, 'x']],
            y=[df_3d_gt.loc[fi, 'y'], df_3d_model.loc[fi, 'y']],
            z=[df_3d_gt.loc[fi, 'z'], df_3d_model.loc[fi, 'z']],
            mode='lines',
            line=dict(color='rgba(255,0,0,0.3)', width=2),
            showlegend=False,
            hoverinfo='skip',
        ))

# 球场
court_x = [0, COURT_WIDTH, COURT_WIDTH, 0, 0]
court_y_line = [0, 0, COURT_LENGTH, COURT_LENGTH, 0]
fig11.add_trace(go.Scatter3d(
    x=court_x, y=court_y_line, z=[0]*5,
    mode='lines', line=dict(color='green', width=3),
    name='球场', showlegend=True,
))

fig11.update_layout(
    title='3D 轨迹: GT（红×）vs 模型（蓝点）',
    scene=dict(
        xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)',
        aspectmode='data',
    ),
    width=900, height=700,
)
fig11.show()

In [ ]:
# --- 图12: 2D 像素级 GT vs 模型对比 (逐摄像头) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, name, cmp in zip(axes, ['cam66', 'cam68'], [cmp_cam66, cmp_cam68]):
    detected = cmp[cmp['pixel_error'].notna()]
    if len(detected) == 0:
        ax.set_title(f'{name}: 无对比数据')
        continue
    
    # GT 点
    ax.scatter(detected['gt_px'], detected['gt_py'], c='red', s=30, marker='x',
               label='GT', zorder=5, linewidths=1.5)
    # 模型点
    ax.scatter(detected['model_px'], detected['model_py'], c='blue', s=15, marker='o',
               label='模型', zorder=4, alpha=0.7)
    # 连线
    for _, row in detected.iterrows():
        color = 'green' if row['pixel_error'] < 10 else 'orange' if row['pixel_error'] < 50 else 'red'
        ax.plot([row['gt_px'], row['model_px']], [row['gt_py'], row['model_py']],
                '-', color=color, linewidth=0.8, alpha=0.5)
    
    ax.set_xlabel('像素 X')
    ax.set_ylabel('像素 Y')
    ax.invert_yaxis()  # 图像坐标系 y 向下
    ax.set_title(f'{name}: GT(红×) vs 模型(蓝○), 平均误差={detected["pixel_error"].mean():.1f}px')
    ax.legend(fontsize=8)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 🔧 实验区

以下区域用于自由实验，调整参数、测试新方法。

In [ ]:
# ===== 在此处自由实验 =====
# 例如：调整分段阈值、尝试不同滤波参数、添加样条插值等

# segments_v2 = segment_trajectory(df_3d, speed_threshold=40.0, gap_threshold=3)
# print(f'调整后分段数: {len(segments_v2)}')